In [2]:
import awkward as ak
import json

In [3]:
df = ak.from_parquet('fixed_data/merged_nominal.parquet')

/home/users/iareed/miniconda3/envs/higgs-dna/lib/python3.10/site-packages/awkward/operations/convert.py:1999: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if distutils.version.LooseVersion(
/home/users/iareed/miniconda3/envs/higgs-dna/lib/python3.10/site-packages/awkward/operations/convert.py:2001: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  ) < distutils.version.LooseVersion("2.0.0"):


In [4]:
with open('fixed_data/summary.json') as f:
    summary = json.load(f)

In [5]:
proc_ids = summary['sample_id_map']

In [6]:
proc_ids

{'Data': 0,
 'DiPhoton': 18,
 'GJets_HT-100To200': 9,
 'GJets_HT-200To400': 10,
 'GJets_HT-400To600': 11,
 'GJets_HT-40To100': 8,
 'GJets_HT-600ToInf': 12,
 'TTGG': 15,
 'TTGamma': 14,
 'TTJets': 13,
 'VBFH_M125': 6,
 'VH_M125': 7,
 'WGamma': 16,
 'ZGamma': 17,
 'ggH_M125': 5,
 'ttHH_ggTauTau': 4,
 'ttHH_ggWW': 3,
 'ttHH_ggbb': 2,
 'ttH_M125': 1}

In [12]:
signal = ['ttHH_ggbb', 'ttHH_ggWW', 'ttHH_ggTauTau']
nonres = ['DiPhoton', 'TTGG', 'TTGamma', 'TTJets', 'WGamma', 'ZGamma']
res = ['VBFH_M125', 'VH_M125', 'ggH_M125', 'ttH_M125']
gjets = ['GJets_HT-40To100', 'GJets_HT-100To200', 'GJets_HT-200To400', 'GJets_HT-400To600', 'GJets_HT-600ToInf']

In [18]:
def group_mask(df, proc_ids, group):
    tmp_mask = df.process_id==-857 #Not -999 incase some error has occured
    for proc in group:
        tmp_mask = tmp_mask | (df.process_id==proc_ids[proc])
    return tmp_mask

In [50]:
nonres_mask = group_mask(df, proc_ids, nonres)
gjets_mask = group_mask(df, proc_ids, gjets)
res_mask = group_mask(df, proc_ids, res)
signal_mask = group_mask(df, proc_ids, signal)
data_mask = df.process_id ==proc_ids['Data']
SR1_mask = df.mva_score>=0.9937 #SR from MC as nonres bkg
#SR1_mask = df.mva_score>=0.9947 #SR from data as nonres bkg

In [40]:
full_nonres_mask = nonres_mask | gjets_mask

In [51]:
gdf = df[gjets_mask & SR1_mask]
nrdf = df[nonres_mask & SR1_mask]
fnrdf = df[full_nonres_mask & SR1_mask]
ddf = df[data_mask & SR1_mask]

In [37]:
def sideband_count(df):
    min_edge = 100
    low_edge = 115
    high_edge = 135
    max_edge = 180
    low_mask = (df.Diphoton_mass >= min_edge) & (df.Diphoton_mass <= low_edge)
    high_mask = (df.Diphoton_mass >= high_edge) & (df.Diphoton_mass <= max_edge)
    low_count = ak.sum(df[low_mask].weight_central)
    high_count = ak.sum(df[high_mask].weight_central)
    print('Num events in the low bin: {}'.format(low_count))
    print('Num events in the high bin: {}'.format(high_count))
    print('----------------------------------')
    print('Num events in the low bin: {:.2f}'.format(low_count))
    print('Num events in the high bin: {:.2f}'.format(high_count))

In [52]:
sideband_count(nrdf)

Num events in the low bin: 1.55424312549341
Num events in the high bin: -1.101810112968988
----------------------------------
Num events in the low bin: 1.55
Num events in the high bin: -1.10


In [53]:
sideband_count(gdf)

Num events in the low bin: 0.0
Num events in the high bin: 0.0
----------------------------------
Num events in the low bin: 0.00
Num events in the high bin: 0.00


In [54]:
sideband_count(fnrdf)

Num events in the low bin: 1.55424312549341
Num events in the high bin: -1.101810112968988
----------------------------------
Num events in the low bin: 1.55
Num events in the high bin: -1.10


In [55]:
sideband_count(ddf)

Num events in the low bin: 4.0
Num events in the high bin: 2.0
----------------------------------
Num events in the low bin: 4.00
Num events in the high bin: 2.00


In [56]:
min_edge = 100
low_edge = 115
high_edge = 135
max_edge = 180
low_mask = (ddf.Diphoton_mass >= min_edge) & (ddf.Diphoton_mass <= low_edge)
high_mask = (ddf.Diphoton_mass >= high_edge) & (ddf.Diphoton_mass <= max_edge)
print(ddf[low_mask].Diphoton_mass)
print(ddf[high_mask].Diphoton_mass)

[112, 102, 100, 102]
[155, 166]
